# Single-jet and dijet closures

This notebook projects selected single-jet and dijet TH2 histograms, aligns all curves to their shared bin edges, applies the configured normalization, and plots both overlays and ratios to an explicitly configured nominal. It supports gen/ref/reco comparisons within one MC generator and embedding/Pythia comparisons at a selected reconstruction level. Direction-specific files and the existing combined-orientation file are read directly.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path('/Users/gnigmat/work/cms/jetAnalysis')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

try:
    import ROOT
except ModuleNotFoundError:
    # Local Homebrew ROOT fallback used by this repository's current environment.
    for path in (Path('/opt/homebrew/lib/python3.14/site-packages'),
                 Path('/opt/homebrew/Cellar/root/6.40.02_1/lib/root')):
        if path.exists() and str(path) not in sys.path:
            sys.path.insert(0, str(path))
    import ROOT

ROOT.gROOT.SetBatch(True)
ROOT.gStyle.SetOptStat(0)

from hist_analysis.config.files import BASE_DIR
from hist_analysis.config.histograms import (
    COMMON_ETA_CM_RANGE, DIJET_PTAVE_BINS, SINGLE_JET_PT_BINS,
    STANDARD_DIJET_ETA_CUT_INDEX,
)
from hist_analysis.python.closures import ClosureCurve, build_closure_histograms
from hist_analysis.python.histogram_io import resolve_combined_file, resolve_direction_file
from hist_analysis.python.plotting import draw_closure

## Configuration

`CURVE_MODE = 'levels'` compares gen/ref/reco in the selected generator and uses `NOMINAL` for the ratio denominator. Use `'samples'` to compare the same `RECO_LEVEL` between embedding and Pythia; in that mode the selected `GENERATOR` is the denominator. `NORMALIZATION` accepts `none`, `integral`, or `bin_width`, and selection intervals are interpreted as `[low, high)`. The configured output consists of PDFs beneath `output/closures/`.

In [ ]:
GENERATOR = 'embedding'       # embedding or pythia
DIRECTIONS = ('pgoing', 'Pbgoing', 'combined')
FILE_STEM = 'jetId'
CURVE_MODE = 'levels'         # levels or samples
RECO_LEVEL = 'Reco'           # used by samples mode: Gen, Ref, or Reco
NOMINAL = 'Gen'               # must exactly match a curve label
NORMALIZATION = 'integral'    # none, integral, or bin_width
RATIO_RANGE = (0.5, 1.5)
OUTPUT_DIR = PROJECT_ROOT / 'hist_analysis' / 'output' / 'closures'

def mc_file(generator, direction):
    if direction == 'combined':
        return resolve_combined_file(BASE_DIR, generator, FILE_STEM)
    return resolve_direction_file(BASE_DIR, generator, direction, FILE_STEM)

def level_curves(direction, jet_kind):
    filename = mc_file(GENERATOR, direction)
    if jet_kind == 'single':
        # New direction files contain CM keys; existing combined files fall back
        # to the already orientation-combined lab-unflipped distributions.
        names = {level: (f'h{level}InclusiveJetPtEtaCM',
                         f'h{level}InclusiveJetPtEtaLabUnflipped')
                 for level in ('Gen', 'Ref', 'Reco')}
    else:
        names = {level: (f'h{level}DijetPtEtaCM_{STANDARD_DIJET_ETA_CUT_INDEX}',)
                 for level in ('Gen', 'Ref', 'Reco')}
    return [ClosureCurve(level, filename, names[level]) for level in ('Gen', 'Ref', 'Reco')]

def sample_curves(direction, jet_kind):
    curves = []
    for generator in ('embedding', 'pythia'):
        if jet_kind == 'single':
            keys = (f'h{RECO_LEVEL}InclusiveJetPtEtaCM',
                    f'h{RECO_LEVEL}InclusiveJetPtEtaLabUnflipped')
        else:
            keys = (f'h{RECO_LEVEL}DijetPtEtaCM_{STANDARD_DIJET_ETA_CUT_INDEX}',)
        curves.append(ClosureCurve(generator, mc_file(generator, direction), keys))
    return curves

def curves_for(direction, jet_kind):
    return level_curves(direction, jet_kind) if CURVE_MODE == 'levels' else sample_curves(direction, jet_kind)

## Single-jet eta closures in configured pT intervals

In [ ]:
single_eta_results = {}
for direction in DIRECTIONS:
    for pt_range in SINGLE_JET_PT_BINS:
        histograms, keys = build_closure_histograms(
            curves_for(direction, 'single'), observable='eta',
            selection_range=pt_range, normalization=NORMALIZATION)
        nominal = NOMINAL if CURVE_MODE == 'levels' else GENERATOR
        tag = f'{direction}_single_eta_pt{pt_range[0]}_{pt_range[1]}'
        canvas, ratios = draw_closure(
            histograms, nominal, title=f'{direction}: single jets, {pt_range[0]} < pT < {pt_range[1]} GeV',
            x_title='#eta_{CM}', ratio_range=RATIO_RANGE,
            output=OUTPUT_DIR / f'{tag}.pdf')
        single_eta_results[tag] = {'canvas': canvas, 'ratios': ratios, 'keys': keys}
        print(tag, keys)

## Single-jet pT closures in the common eta range

In [ ]:
single_pt_results = {}
for direction in DIRECTIONS:
    histograms, keys = build_closure_histograms(
        curves_for(direction, 'single'), observable='pt',
        selection_range=COMMON_ETA_CM_RANGE, normalization=NORMALIZATION)
    nominal = NOMINAL if CURVE_MODE == 'levels' else GENERATOR
    tag = f'{direction}_single_pt'
    canvas, ratios = draw_closure(
        histograms, nominal, title=f'{direction}: single jets, |eta_CM| < 1.9',
        x_title='p_{T} (GeV)', ratio_range=RATIO_RANGE, log_y=True,
        output=OUTPUT_DIR / f'{tag}.pdf')
    single_pt_results[tag] = {'canvas': canvas, 'ratios': ratios, 'keys': keys}
    print(tag, keys)

## Dijet eta closures in configured pTave intervals (standard eta-cut index 5)

In [ ]:
dijet_eta_results = {}
for direction in DIRECTIONS:
    for ptave_range in DIJET_PTAVE_BINS:
        histograms, keys = build_closure_histograms(
            curves_for(direction, 'dijet'), observable='eta',
            selection_range=ptave_range, normalization=NORMALIZATION)
        nominal = NOMINAL if CURVE_MODE == 'levels' else GENERATOR
        tag = f'{direction}_dijet_eta_ptave{ptave_range[0]}_{ptave_range[1]}'
        canvas, ratios = draw_closure(
            histograms, nominal, title=f'{direction}: dijets, {ptave_range[0]} < pTave < {ptave_range[1]} GeV',
            x_title='#eta_{dijet,CM}', ratio_range=RATIO_RANGE,
            output=OUTPUT_DIR / f'{tag}.pdf')
        dijet_eta_results[tag] = {'canvas': canvas, 'ratios': ratios, 'keys': keys}
        print(tag, keys)

## Dijet pTave closures in the common eta range

In [ ]:
dijet_pt_results = {}
for direction in DIRECTIONS:
    histograms, keys = build_closure_histograms(
        curves_for(direction, 'dijet'), observable='ptave',
        selection_range=COMMON_ETA_CM_RANGE, normalization=NORMALIZATION)
    nominal = NOMINAL if CURVE_MODE == 'levels' else GENERATOR
    tag = f'{direction}_dijet_ptave'
    canvas, ratios = draw_closure(
        histograms, nominal, title=f'{direction}: dijets, |eta_dijet,CM| < 1.9',
        x_title='p_{T}^{ave} (GeV)', ratio_range=RATIO_RANGE, log_y=True,
        output=OUTPUT_DIR / f'{tag}.pdf')
    dijet_pt_results[tag] = {'canvas': canvas, 'ratios': ratios, 'keys': keys}
    print(tag, keys)

## Adding data or another explicit nominal

Construct a list such as `ClosureCurve('data', Path('/exact/file.root'), ('histogramKey',))`, pass it to `build_closure_histograms`, and set `nominal='data'` in `draw_closure`. A curve may provide multiple candidate keys; the first existing key is loaded and recorded in the result. Data trigger merging, frame conventions, and normalization are analysis choices and are intentionally not inferred by this notebook.